# D02 — Données et audit de GLD

Ce notebook présente les données utilisées dans notre projet,
leur provenance et les contrôles de qualité.

- Actif : GLD — SPDR Gold Shares.
- Source : Yahoo Finance via yfinance.
- Fréquence : journalière.
- Période demandée : du 01/01/2018 au 30/10/2026 inclus.
- Prix OHLC ajustés avec auto_adjust=True.

Les données sont chargées depuis un snapshot local daté.
Aucun modèle n'est entraîné dans ce notebook.

## 1. Préparer l’environnement et importer les fonctions

Cette étape permet au notebook d’utiliser les fonctions définies dans
`src/data.py`, sans recopier leur code.

Le code :
- Repère le dossier depuis lequel le notebook est exécuté avec `Path.cwd()`.
- Recherche la racine du projet, qui contient `src/data.py`.
- Ajoute cette racine à `sys.path` si elle n’y figure pas déjà.
  `sys.path` est la liste des emplacements où Python recherche les modules.
- Importe `load_snapshot()` pour lire les données et `audit_data()`
  pour vérifier leur qualité.

Si le projet n’est pas trouvé, une erreur explicite est affichée.

**Résultat attendu :** le chemin de la racine du projet s’affiche.


In [1]:
from pathlib import Path
import sys
import json

# Le notebook peut démarrer depuis notebooks/ ou depuis la racine.
current_folder = Path.cwd()

if (current_folder / "src" / "data.py").exists():
    PROJECT_ROOT = current_folder
elif (current_folder.parent / "src" / "data.py").exists():
    PROJECT_ROOT = current_folder.parent
else:
    raise FileNotFoundError(
        "Ouvre le notebook depuis le dossier du projet ou notebooks/."
    )

# Permet à Python de trouver notre package src.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_snapshot, audit_data

print("Racine du projet :", PROJECT_ROOT)

Racine du projet : c:\Users\ilyas\Desktop\idld\idldS3\trading intelligent\GLD Quant Lab


## 2. Identifier les snapshots disponibles

Un snapshot est une copie locale des données, conservée dans un dossier daté.
Il permet de reproduire l’analyse sans refaire une requête à Yahoo Finance.

Le code :
- Recherche les fichiers `GLD_daily.csv` dans les sous-dossiers `GLD_*`
  du dossier `data/raw/`.
- Trie les chemins trouvés pour obtenir un affichage stable.
- Affiche leur emplacement par rapport à la racine du projet.

Si aucun fichier n’est trouvé, le programme indique qu’il faut d’abord
exécuter le script de collecte avec `python -m src.data`.

**Résultat attendu :** la liste des snapshots disponibles.

Cette étape liste les fichiers ; elle ne charge pas encore leur contenu.

In [2]:
snapshots = sorted(
    (PROJECT_ROOT / "data" / "raw").glob("GLD_*/GLD_daily.csv")
)

if not snapshots:
    raise FileNotFoundError(
        "Aucun snapshot trouvé. Lance d'abord : python -m src.data"
    )

for path in snapshots:
    print(path.relative_to(PROJECT_ROOT))

data\raw\GLD_20261115T154114487177Z\GLD_daily.csv


## 3. Choisir et charger les données

Nous sélectionnons explicitement un snapshot parmi les fichiers disponibles.

Le code :
- Enregistre le chemin choisi dans la variable `SNAPSHOT`.
- Construit son chemin complet à partir de la racine du projet.
- Appelle `load_snapshot()` pour lire le CSV dans un tableau pandas nommé `df`.
- Convertit la colonne `Date` en dates grâce à la fonction de chargement.
- Affiche les cinq premières lignes avec `df.head()`.

Il faut remplacer le chemin d’exemple par celui d’un fichier réellement présent.

**Pourquoi choisir un fichier précis ?**
Une nouvelle collecte ne doit pas changer automatiquement les données
utilisées par ce notebook.

**Résultat attendu :** les cinq premières observations du snapshot.
Aucune donnée n’est téléchargée ou nettoyée à cette étape.

In [4]:
SNAPSHOT = "data/raw/GLD_20261115T154114487177Z/GLD_daily.csv"

csv_path = PROJECT_ROOT / SNAPSHOT

df = load_snapshot(csv_path)

df.head()

,Date,Capital Gains,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2018-01-02,0.0,125.150002,0.0,125.180000,124.389999,124.660004,0.0,11762500
1,2018-01-03,0.0,124.820000,0.0,125.089996,124.099998,125.050003,0.0,7904300
2,2018-01-04,0.0,125.459999,0.0,125.849998,124.739998,124.889999,0.0,7329700
3,2018-01-05,0.0,125.330002,0.0,125.480003,124.830002,124.930000,0.0,5739900
4,2018-01-08,0.0,125.309998,0.0,125.320000,124.900002,125.199997,0.0,3566700


## 4. Vérifier la provenance des données

Les métadonnées décrivent l’origine et les conditions de récupération
du snapshot. Elles sont enregistrées dans `metadata.json`, à côté du CSV.

Le code :
- Localise ce fichier dans le dossier du snapshot sélectionné.
- Ouvre le fichier en lecture avec l’encodage UTF-8.
- Utilise `json.load()` pour convertir son contenu en dictionnaire Python.
- Affiche la source, l’actif, la date d’acquisition, la période observée,
  le nombre de lignes et la convention d’ajustement.

**Résultat attendu :** les informations de provenance enregistrées
au moment de la collecte.

La période observée peut différer légèrement de la période demandée,
notamment lorsque les dates de début ou de fin ne sont pas des séances
boursières.

**Limite :** afficher les métadonnées ne vérifie pas que le CSV
n’a pas été modifié depuis sa création.

In [5]:
metadata_path = csv_path.parent / "metadata.json"

with metadata_path.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

print("Source :", metadata["source"])
print("Actif :", metadata["symbol"])
print("Date d'acquisition UTC :", metadata["acquired_at_utc"])
print("Première séance :", metadata["observed_first_date"])
print("Dernière séance :", metadata["observed_last_date"])
print("Nombre de lignes :", metadata["rows"])
print("Ajustement :", metadata["adjustment_convention"])

Source : Yahoo Finance via yfinance
Actif : GLD
Date d'acquisition UTC : 2026-11-15T15:41:14.487177+00:00
Première séance : 2018-01-02
Dernière séance : 2026-10-30
Nombre de lignes : 2220
Ajustement : OHLC ajustés par yfinance


## 5. Réaliser les premiers contrôles qualité

Avant d’utiliser les données, nous recherchons les anomalies qui pourraient
fausser les calculs et les futurs modèles.

La fonction `audit_data(df)` vérifie :
- Les lignes contenant au moins une valeur manquante.
- Les dates dupliquées supplémentaires.
- Le respect de l’ordre chronologique.
- Les prix nuls ou négatifs.
- La cohérence de High et Low avec les autres prix de la séance.
- Les volumes négatifs.
- Les volumes nuls, à examiner.

Le résultat est enregistré dans `quality_report`, puis affiché sous forme
de tableau.

**Comment interpréter les résultats ?**
- Un compteur à 0 signifie que le contrôle n’a détecté aucun cas.
- Un compteur positif indique des observations à examiner.
- Pour l’ordre chronologique, 0 signifie que l’ordre est respecté et 1
  qu’il ne l’est pas : ce n’est pas un nombre de lignes.
- Une même ligne peut déclencher plusieurs contrôles. Il ne faut donc pas
  additionner tous les compteurs pour obtenir un nombre de lignes incorrectes.

**Résultat attendu :** un tableau des premiers contrôles qualité.

Cette étape ne supprime et ne corrige aucune donnée. Elle ne vérifie pas
encore les séances manquantes ni toutes les anomalies possibles.

In [6]:
quality_report = audit_data(df)

quality_report

,Controle,Nombre
0,Lignes avec au moins une valeur manquante,0
1,Dates dupliquées supplémentaires,0
2,Ordre chronologique incorrect,0
3,Lignes avec un prix nul ou négatif,0
4,Lignes avec un High incohérent,0
5,Lignes avec un Low incohérent,0
6,Lignes avec un volume négatif,0
7,Lignes avec un volume nul — à examiner,0
